# Using ``AddAccis()`` to apply adaptive setpoint temperatures

We are going to apply the adaptive setpoint temperatures to the IDF "TestModel_onlyGeometryForVRFsystem_2zones_CalcVent_V2510" (a model for EnergyPlus 25.1), which can be found in the folder ``sample IDFs/input_IDFs`` of the sample files. For clarity purposes, it has been renamed to "TestModel.idf". ``AddAccis()`` does not run any simulation, but it needs the EnergyPlus version of the IDF (25.1) to be installed, since it reads its IDD file.

``AddAccis()`` was named ``addAccis()`` before accim 1.0.0, and its arguments had other names (``ScriptType`` instead of ``script_type``, ``ComfStand`` instead of ``comfort_standard``...): the old names still work, with a deprecation warning, and the questions that it asks still show them.

Since accim 1.1.0, ``AddAccis()`` prints the estimated size of the results of the simulations of the output IDFs before writing them (section 2.2.2), and each output IDF only has the EMS code of the comfort standards of its zones (section 2.1). Section 2.3 shows how to read the EMS code with which the ACCIS calculates the setpoints of a comfort standard before adding it to a model, and section 2.4 how to change the arguments of an output IDF, also its comfort standard, with ``modify_accis()``.

First of all, let's see what files we do have in the folder:

In [ ]:
import os
os.listdir()

You can see there is/are a/some IDF file/files there, which is/are:

In [ ]:
input_idf = [i for i in os.listdir() if i.endswith('.idf')]
print(*input_idf, sep='\n')

So, let's apply adaptive setpoint temperatures.

## 1. General use

### 1.1 Short version

First, let's do this using the short version. When we run the code below (not entering any argument), accim will request some information to generate the output idfs. We are going to enter the following information in **bold**:

Enter the ScriptType (
for VRFsystem with full air-conditioning mode: vrf_ac;
for VRFsystem with mixed-mode: vrf_mm;
for ExistingHVAC with mixed mode: ex_mm;
for ExistingHVAC with full air-conditioning mode: ex_ac
): **vrf_mm**

Enter the SupplyAirTempInputMethod (
for Supply Air Temperature: supply air temperature;
for Temperature Difference: temperature difference;
): **temperature difference**

Do you want to keep the existing outputs (true or false)?: **false**

Enter the Output type (standard, simplified, detailed or custom): **standard**

Enter the Output frequencies separated by space (timestep, hourly, daily, monthly, runperiod): **hourly runperiod**

Do you want to generate a dataframe to see all outputs? (true or false): **false**

Enter the EnergyPlus version (9.1 to 26.2, or auto): **auto** (accim reads the version from the IDF, 25.1 in this case)

Enter the Temperature Control method (temperature or pmv): **temperature**

Then, accim will report the information we just entered, and will start generating a generic idf file, which will be modified based on the remaining arguments accim will request in a second stage. Again, we are going to enter the following information in **bold**:

Enter the Comfort Standard numbers separated by space (...): **1 14**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the Category numbers separated by space (...): **1 2 3 80 90**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the Comfort Mode numbers separated by space (...): **0 0.1 0.2 0.3 0.4 0.5 1 1.1 1.2 1.3 1.4 1.5 2 3**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the setpoint accuracy number (any number greater than 0, if omitted will be 10000): **100**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the number for the CAT cooling offset modifier (value will be summed to the ACST, if omitted will be 0): **0**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the number for the CAT heating offset modifier (value will be summed to the AHST, if omitted will be 0): **0**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the start of the cooling season in numeric date format dd/mm or the day of the year: **01/11**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the end of the cooling season in numeric date format dd/mm or the day of the year: **01/03**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the HVAC Mode numbers separated by space (...): **2**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)** 

Enter the Ventilation Control numbers separated by space (...): **2**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)** 

Enter the maximum temperature difference number for Ventilation Opening Factor (any number larger than 0): **20**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the minimum temperature difference number for Ventilation Opening Factor (any number larger than 0 and smaller than the maximum temperature difference number): **1**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the multiplier number for Ventilation Opening Factor (any number between 0 and 1): **0.2**

Are you sure the number is correct? [y or [] / n]: **(just hit enter)**

Enter the VSToffset numbers separated by space (if omitted, will be 0): **(just hit enter)**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the MinOToffset numbers separated by space (if omitted, will be 50): **(just hit enter)**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the MaxWindSpeed numbers separated by space (if omitted, will be 50): **(just hit enter)**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the ASTtol value from (if omitted, will be 0.1): **(just hit enter)**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the ASTtol value to (if omitted, will be 0.1): **(just hit enter)**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

Enter the ASTtol value steps (if omitted, will be 0.1): **(just hit enter)**

Are you sure the numbers are correct? [y or [] / n]: **(just hit enter)**

And finally, after listing the output IDFs and the estimated size of the results of their simulations (see section 2.2.2):

Do you still want to run ACCIS? [y/n]: **y**

In [ ]:
from accim.sim import accis
accis.AddAccis()

Let's see what files we do have now in the folder:

In [ ]:
os.listdir()

We can see there are new IDFs, as stated previously in the python console:

In [ ]:
output_idfs = [i for i in os.listdir() if not(any(i in j for j in input_idf)) and i.endswith('.idf')]
print(*output_idfs, sep='\n')

Now, these are ready for simulation with any EPW file, although it is recommended to have previously renamed them with the class rename_epw_files().

Let's remove the new output IDFs for clarity purposes:

In [ ]:
for i in os.listdir():
    if i.endswith('.idf') and input_idf[0] not in i:
        os.remove(i)

### 1.2 Longer version

Let's double check we only have the initial idf we used in the short version:

In [ ]:
input_idf = [i for i in os.listdir() if i.endswith('.idf')]
print(*input_idf, sep='\n')

Now, let's do the same, but using the longer version:

In [ ]:
from accim.sim import accis
accis.AddAccis(
    script_type='vrf_mm',
    supply_air_temp_input_method='temperature difference',
    keep_existing_outputs=False,
    output_type='standard',
    output_frequencies=['hourly', 'runperiod'],
    make_outputs_dataframe=False,
    energyplus_version='auto',
    setpoint_type='temperature',
    comfort_standard=[1, 14],
    category=[1, 2, 3, 80, 90],
    category_cooling_offset=0,
    category_heating_offset=0,
    comfort_mode=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 1.1, 1.2, 1.3, 1.4, 1.5, 2, 3],
    setpoint_accuracy=100,
    cooling_season_start='01/11',
    cooling_season_end='01/03',
    hvac_mode=[2],
    ventilation_control=[2],
    opening_factor_max_temp_diff=20,
    opening_factor_min_temp_diff=1,
    min_opening_factor=0.2,
    ventilation_setpoint_offset=[0],
    min_outdoor_temp_offset=[50],
    max_wind_speed=[50],
    setpoint_tolerance_step=0.1,
    setpoint_tolerance_start=0.1,
    setpoint_tolerance_end=0.1,
    confirm=True
)

Let's see what files we do have now in the folder:

In [ ]:
os.listdir()

We can see there are new IDFs, as stated previously in the python console:

In [ ]:
output_idfs = [i for i in os.listdir() if not(any(i in j for j in input_idf)) and i.endswith('.idf')]
print(*output_idfs, sep='\n')

Now, these are ready for simulation with any EPW file, although it is recommended to have previously renamed them with the class rename_epw_files().

Let's remove the new output IDFs for clarity purposes:

In [ ]:
for i in os.listdir():
    if i.endswith('.idf') and input_idf[0] not in i:
        os.remove(i)

## 2. Other uses

### 2.1 Storing the ``AddAccis()`` class and inspecting the output IDFs

Additionally to just run ``AddAccis()`` (i.e. instantiate the class), the user can also store the instance and inspect some variables. So let's store the instance in the variable ``add_accis_instance`` and then we'll take a look what's inside:

In [ ]:
from accim.sim import accis
add_accis_instance = accis.AddAccis(
    script_type='vrf_mm',
    supply_air_temp_input_method='temperature difference',
    keep_existing_outputs=False,
    output_type='standard',
    output_frequencies=['hourly', 'runperiod'],
    make_outputs_dataframe=False,
    energyplus_version='auto',
    setpoint_type='temperature',
    comfort_standard=[1, 14],
    category=[1, 2, 3, 80, 90],
    comfort_mode=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 1.1, 1.2, 1.3, 1.4, 1.5, 2, 3],
    setpoint_accuracy=100,
    cooling_season_start='01/11',
    cooling_season_end='01/03',
    hvac_mode=[2],
    ventilation_control=[2],
    opening_factor_max_temp_diff=20,
    opening_factor_min_temp_diff=1,
    min_opening_factor=0.2,
    ventilation_setpoint_offset=[0],
    min_outdoor_temp_offset=[50],
    max_wind_speed=[50],
    setpoint_tolerance_step=0.1,
    setpoint_tolerance_start=0.1,
    setpoint_tolerance_end=0.1,
    confirm=True
)

Now, let's take a look into the variables within the ``AddAccis()`` instance ``add_accis_instance``:

In [ ]:
[i for i in dir(add_accis_instance) if not i.startswith('_')]

For instance, we can take a look at the arguments we have requested when ``AddAccis`` was called:

In [ ]:
add_accis_instance.arguments

Also, we can see the list of occupied zones and windows and doors in each model, considering the original names and the new names needed by accim:

In [ ]:
add_accis_instance.occupied_zones

In [ ]:
add_accis_instance.occupied_zones_original_name

In [ ]:
add_accis_instance.windows_and_doors

In [ ]:
add_accis_instance.windows_and_doors_original_name

As you can see, these variables are dictionaries following the pattern: ``{'input idf filename': [list]}``

Also, we can inspect both the input and output idf files using eppy. The variable ``input_idfs`` is a dictionary following the pattern ``{'idf filename': eppy.modeleditor.IDF object}``, and ``output_idfs`` works like one, although it reads each output IDF when it is used (``output_idfs.paths`` gives the path of each file). For instance, let's take a look at the ``input_idfs`` variable.

In [ ]:
add_accis_instance.input_idfs

So let's read the eppy.modeleditor.IDF object of the idf TestModel. For instance, let's see which zones are in the IDF by reading it, and if these are the same as previously seen in ``add_accis_instance.occupied_zones_original_name``:

In [ ]:
testmodel = add_accis_instance.input_idfs['TestModel']
testmodel.idfobjects['ZONE']

And we can see these are the same. Also, you can take a look at the variable ``output_idfs``, which is another dictionary similar to ``input_idfs`` but with the output files.

In [ ]:
add_accis_instance.output_idfs

Let's read the last one. Let's see, for instance, the Schedule:Compact objects for the thermostats:

In [ ]:
ast_testmodel = add_accis_instance.output_idfs[
    'TestModel[CS_AUS DeDear[CA_90[CM_3[HM_2[VC_2[VO_0.0[MT_50.0[MW_50.0[AT_0.1[NS_X.idf'
]
ast_testmodel.idfobjects['ThermostatSetpoint:DualSetpoint']

And now let's compare this with the input idf:

In [ ]:
testmodel.idfobjects['ThermostatSetpoint:DualSetpoint']

Therefore, we can confirm the Schedule:Compact objects for output idf stored in variable ``ast_testmodel`` are AHST and ACST, and thus, adaptive setpoint temperatures have been applied. However, this is different in the original idf, stored in variable ``testmodel``, in which setpoint temperatures are static.

Each output IDF only has the EMS code of the comfort standards of its zones. For instance, the subprograms that calculate the setpoints of ``ast_testmodel`` (``SetAST_CS<n>_<chunk>``) are only those of AUS DeDear (``comfort_standard=14``), its comfort standard. Before accim 1.1.0, the output IDFs had the EMS code of the 24 comfort standards: this one had 78 EMS programs.

In [ ]:
programs = ast_testmodel.idfobjects['EnergyManagementSystem:Program']
print(len(programs), 'EMS programs')
[program.Name for program in programs if program.Name.startswith('SetAST_CS')]

To change the comfort standard of an output IDF, use ``modify_accis()``, which adds the EMS code of the comfort standards that it sets (section 2.4).

For more information about the use of eppy, please refer to the documentation at https://eppy.readthedocs.io/en/latest/ or https://eppy.readthedocs.io/en/latest/Main_Tutorial.html

Let's delete all the output idf files so that we can re-run the notebook.

In [ ]:
for i in os.listdir():
    if i.endswith('.idf') and input_idf[0] not in i:
        os.remove(i)

### 2.2 Specifying the outputs in the IDF

#### 2.2.1 Using ``output_type='custom'``

Imagine you need to perform a large number of simulations, therefore, using ``output_type='standard'`` would result in a heavy dataset (``AddAccis()`` prints the estimated size of the results before writing the output IDFs, see section 2.2.2). Using ``output_type='simplified'`` is not an option, since you need more outputs than just the operative temperature, heating coil heating rate and cooling coil total cooling rate. In that case, you can use ``output_type='custom'``. If you do so, accim will iterate through the existing reporting frequencies and will show you all the existing Output:Variable objects for that frequency, and then, you'll be asked the following:
- firstly, you'll need to say if you want to remove some object or otherwise keep some and remove all others by entering "remove" or "keep"
- secondly, you'll have to say all Output:Variable objects you want to remove or keep

So let's say we want to keep (therefore, enter **keep**) the following Output:Variable objects (you may not be aware of them, but in that case, you can see the existing objects in the list accim prints), which you'll need to enter separated by semicolon (;):

**Zone Operative Temperature;Adaptive Cooling Setpoint Temperature_No Tolerance;Adaptive Heating Setpoint Temperature_No Tolerance**

Also, in this case, we are going to store the ``AddAccis()`` instance in the variable ``add_accis_instance`` (similar to previous section 2.1), so that we can inspect it later.

First of all, let's see the idfs we do have in this path:

In [ ]:
from os import listdir
input_idfs = [i for i in listdir() if i.endswith('.idf')]
input_idfs

Now, we can go for it:

In [ ]:
from accim.sim import accis
add_accis_instance = accis.AddAccis(
    script_type='vrf_mm',
    supply_air_temp_input_method='temperature difference',
    keep_existing_outputs=False,
    output_type='custom',
    output_frequencies=['hourly', 'runperiod'],
    make_outputs_dataframe=False,
    energyplus_version='auto',
    setpoint_type='temperature',
    comfort_standard=[1, 14],
    category=[1, 2, 3, 80, 90],
    comfort_mode=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 1.1, 1.2, 1.3, 1.4, 1.5, 2, 3],
    setpoint_accuracy=100,
    cooling_season_start='01/11',
    cooling_season_end='01/03',
    hvac_mode=[2],
    ventilation_control=[2],
    opening_factor_max_temp_diff=20,
    opening_factor_min_temp_diff=1,
    min_opening_factor=0.2,
    ventilation_setpoint_offset=[0],
    min_outdoor_temp_offset=[50],
    max_wind_speed=[50],
    setpoint_tolerance_step=0.1,
    setpoint_tolerance_start=0.1,
    setpoint_tolerance_end=0.1,
    confirm=True
)

Now, let's inspect the Output:Variable objects in both the input idf and any of the output idfs, for instance, the first one.

In [ ]:
input_testmodel = add_accis_instance.input_idfs['TestModel']
input_testmodel.idfobjects['Output:Variable']

We can see there is no Output:Variable object in the input idf. Now let's see some output idf.

In [ ]:
output_testmodel = add_accis_instance.output_idfs[
    'TestModel[CS_INT EN16798[CA_1[CM_0[HM_2[VC_2[VO_0.0[MT_50.0[MW_50.0[AT_0.1[NS_X.idf'
]
output_testmodel.idfobjects['Output:Variable']

We can see in this case 6 Output:Variable objects, which are the ones we previously requested:
- Zone Operative Temperature
- Adaptive Cooling Setpoint Temperature_No Tolerance
- Adaptive Heating Setpoint Temperature_No Tolerance

in both Hourly and Runperiod Reporting Frequencies.

As usual, let's remove all the output idfs, so that we can re-run the notebook.

In [ ]:
for i in os.listdir():
    if i.endswith('.idf') and input_idf[0] not in i:
        os.remove(i)

#### 2.2.2 Using ``make_outputs_dataframe`` and ``outputs_dataframe``

Similar to section 2.2.1, imagine you need to perform a large number of simulations. Therefore, using output_type='standard' would result in a heavy dataset. Using output_type='simplified' is not an option, since you need more outputs than just the operative temperature, heating coil heating rate and cooling coil total cooling rate. In that case, you can use ``make_outputs_dataframe`` and ``outputs_dataframe``.

First of all, let's see the idfs we do have in this path:

In [ ]:
from os import listdir
input_idfs = [i for i in listdir() if i.endswith('.idf')]
input_idfs

Now, we can go for it. Let's apply the adaptive setpoint temperatures. We want to see what outputs we would have if we run this script, therefore, we need to specify ``make_outputs_dataframe=True``. Since we don't want to generate the output idfs yet, in this case let's specify ``confirm=False``.

In [ ]:
from accim.sim import accis
add_accis_instance = accis.AddAccis(
    script_type='vrf_mm',
    supply_air_temp_input_method='temperature difference',
    keep_existing_outputs=False,
    output_type='standard',
    output_frequencies=['hourly', 'runperiod'],
    make_outputs_dataframe=True,
    energyplus_version='auto',
    setpoint_type='temperature',
    comfort_standard=[1, 14],
    category=[1, 2, 3, 80, 90],
    comfort_mode=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 1.1, 1.2, 1.3, 1.4, 1.5, 2, 3],
    setpoint_accuracy=100,
    cooling_season_start='01/11',
    cooling_season_end='01/03',
    hvac_mode=[2],
    ventilation_control=[2],
    opening_factor_max_temp_diff=20,
    opening_factor_min_temp_diff=1,
    min_opening_factor=0.2,
    ventilation_setpoint_offset=[0],
    min_outdoor_temp_offset=[50],
    max_wind_speed=[50],
    setpoint_tolerance_step=0.1,
    setpoint_tolerance_start=0.1,
    setpoint_tolerance_end=0.1,
    confirm=False
)

We can see no output idf has been generated:

In [ ]:
[i for i in listdir() if i.endswith('.idf')]

Before asking for the confirmation (here, ``confirm=False``), ``AddAccis()`` printed the estimated size of the results of the simulations of the output IDFs (see the output of the cell above): about 31 MB for each simulation with the standard outputs at hourly and run period frequencies, that is, about 1.1 GB for the 36 output IDFs with each EPW. It is estimated from the outputs of the model (the values that its ``Output:Variable`` and ``Output:Meter`` objects report, and how often), without simulating it, so that we can choose other outputs before writing the output IDFs. ``AddAccis()`` keeps it in the variable ``results_size``, in MB, with a row for each input IDF: the size of each file of a simulation of one of its output IDFs (``eso_mb``, ``csv_mb``, ``mtr_mb``, ``meter_csv_mb``, ``sql_mb`` and ``other_mb``), of the whole simulation (``simulation_mb``), of the files that ``accim.run.remove_files()`` keeps (``kept_mb``), the memory needed to read the values of its CSV file (``csv_memory_mb``), and the size of the simulations of all its output IDFs with one EPW (``total_mb`` and ``total_kept_mb``). To measure it in a simulation, see ``estimate_results_size()`` in the notebook about ``run_ep()``.

In [ ]:
results_size_standard = add_accis_instance.results_size
results_size_standard

Since we specified ``make_outputs_dataframe=True``, there is also a variable within ``add_accis_instance`` named ``df_outputs``, which is a pandas DataFrame that contains all Output:Variable objects. Let's take a look:

In [ ]:
add_accis_instance.df_outputs

Now, let's save it in a new variable, and let's filter it to keep only the Output:Variable objects we need.

In [ ]:
df_outputs_in = add_accis_instance.df_outputs
df_outputs_in = df_outputs_in[
    (
        df_outputs_in['variable_name'].str.contains('Zone Operative Temperature')
        |
        df_outputs_in['variable_name'].str.contains('Adaptive Cooling Setpoint Temperature_No Tolerance')
        |
        df_outputs_in['variable_name'].str.contains('Adaptive Heating Setpoint Temperature_No Tolerance')
    )
]
df_outputs_in


Now, we can run again ``AddAccis``, specifying ``make_outputs_dataframe=False``, entering the pandas DataFrame instance ``df_outputs_in`` we just filtered in the argument ``outputs_dataframe``, and specifying ``confirm=True``, since now we're ready to apply the adaptive setpoints.

In [ ]:
from accim.sim import accis
add_accis_instance = accis.AddAccis(
    script_type='vrf_mm',
    supply_air_temp_input_method='temperature difference',
    keep_existing_outputs=False,
    output_type='standard',
    output_frequencies=['hourly', 'runperiod'],
    make_outputs_dataframe=False,
    outputs_dataframe=df_outputs_in,
    energyplus_version='auto',
    setpoint_type='temperature',
    comfort_standard=[1, 14],
    category=[1, 2, 3, 80, 90],
    comfort_mode=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1, 1.1, 1.2, 1.3, 1.4, 1.5, 2, 3],
    setpoint_accuracy=100,
    cooling_season_start='01/11',
    cooling_season_end='01/03',
    hvac_mode=[2],
    ventilation_control=[2],
    opening_factor_max_temp_diff=20,
    opening_factor_min_temp_diff=1,
    min_opening_factor=0.2,
    ventilation_setpoint_offset=[0],
    min_outdoor_temp_offset=[50],
    max_wind_speed=[50],
    setpoint_tolerance_step=0.1,
    setpoint_tolerance_start=0.1,
    setpoint_tolerance_end=0.1,
    confirm=True
)

Before writing the output IDFs, ``AddAccis()`` printed the estimated size of their results again. With these outputs, it is much smaller than with the standard outputs: about 6.7 MB for each simulation instead of 31 MB, that is, about 240 MB for the 36 output IDFs with each EPW instead of 1.1 GB (the ``Output:Meter`` objects, which ``outputs_dataframe`` does not filter, still write the MTR and Meter.csv files):

In [ ]:
import pandas as pd
pd.concat(
    [results_size_standard, add_accis_instance.results_size],
    keys=['standard outputs', 'filtered outputs']
)

Now, let's inspect the Output:Variable objects in both the input idf and any of the output idfs, for instance, the first one.

In [ ]:
input_testmodel = add_accis_instance.input_idfs['TestModel']
input_testmodel.idfobjects['Output:Variable']

We can see there is no Output:Variable object in the input idf. Now let's see some output idf.

In [ ]:
output_testmodel = add_accis_instance.output_idfs[
    'TestModel[CS_INT EN16798[CA_1[CM_0[HM_2[VC_2[VO_0.0[MT_50.0[MW_50.0[AT_0.1[NS_X.idf'
]
output_testmodel.idfobjects['Output:Variable']

We can see in this case 6 Output:Variable objects, which are the ones we previously requested:
- Zone Operative Temperature
- Adaptive Cooling Setpoint Temperature_No Tolerance
- Adaptive Heating Setpoint Temperature_No Tolerance

in both Hourly and Runperiod Reporting Frequencies.

As usual, let's remove all the output idfs, so that we can re-run the notebook.

In [ ]:
for i in os.listdir():
    if i.endswith('.idf') and input_idf[0] not in i:
        os.remove(i)

### 2.3 Reading the EMS code of the setpoints before adding the ACCIS

Before adding the ACCIS to a model, you may want to know how it calculates the setpoints of a comfort model: which setpoints apply, and what happens out of the applicability limits of the comfort model. ``print_setpoint_ems_code()``, in ``accim.utils``, prints the EMS code (in Erl, the language of the Energy Management System of EnergyPlus) with which the ACCIS calculates the setpoints of a comfort standard, category and comfort mode. It is the code of the output IDFs, indented, in the order in which the ACCIS runs it in each timestep, with comments that explain the variables. For instance, for EN 16798-1 (``comfort_standard=1``), category II (``category=2``) and adaptive setpoints, horizontally extended out of the applicability limits (``comfort_mode=3``):

In [ ]:
from accim.utils import print_setpoint_ems_code
print_setpoint_ems_code(comfort_standard=1, category=2, comfort_mode=3)

The setpoints are calculated in the section of the subprogram ``SetAST_CS1_1``: when the running mean outdoor temperature (``RMOT``) is within the applicability limits (10 to 30 °C: ``ACSTall`` and ``ACSTaul`` for the cooling setpoint, ``AHSTall`` and ``AHSTaul`` for the heating setpoint), the cooling and heating setpoints (``ACST`` and ``AHST``) follow it; out of them, the setpoints keep the value that they have at the limit. Then they are rounded with the accuracy ``SetpointAcc``.

``print_setpoint_ems_code()`` takes the other arguments of ``modify_accis()``, with one value each (``setpoint_tolerance``, ``cooling_season_start``, ``category_cooling_offset``... and those of the custom comfort model, ``comfort_standard=99``), and rejects the combinations that give no setpoints, as ``AddAccis()`` does. Without ``category`` or ``comfort_mode``, it prints the code of all of them. ``get_setpoint_ems_code()`` gives the same code as text, to save it in a file or to search in it. For instance, let's print only the lines of the setpoints of comfort mode 2, whose setpoints are static out of the applicability limits:

In [ ]:
from accim.utils import get_setpoint_ems_code
code = get_setpoint_ems_code(comfort_standard=1, category=2, comfort_mode=2)
# from the header of its section to the rounding of the setpoints
print(code[code.index('! --- SetAST_CS1_'):code.index('set ACSTx2')])

In this case, below the lower applicability limit, the setpoints of category II are 25 °C (cooling) and 20 °C (heating), and above the upper limit, 26 °C and 23 °C.

### 2.4 Changing the arguments of an output IDF with ``modify_accis()``

Each zone of an output IDF has the arguments of the ACCIS in its EMS program ``SetInputData_<zone>``, where they can be edited (for instance, with the IDF Editor of EnergyPlus), except the comfort standard: since each output IDF only has the EMS code of the comfort standards of its zones, EnergyPlus would stop with a comfort standard without its code. ``modify_accis()``, in ``accim.sim.accis``, sets the arguments of the ACCIS of an IDF object, in all its occupied zones or, with ``zone_arguments``, in some of them, and adds the EMS code of the comfort standards that it sets.

Let's generate an output IDF with EN 16798-1, category II and comfort mode 3 (``verbose=False`` prints less information):

In [ ]:
from accim.sim import accis
accis.AddAccis(
    script_type='vrf_mm',
    supply_air_temp_input_method='temperature difference',
    keep_existing_outputs=False,
    output_type='standard',
    output_frequencies=['hourly', 'runperiod'],
    energyplus_version='auto',
    setpoint_type='temperature',
    comfort_standard=[1],
    category=[2],
    comfort_mode=[3],
    hvac_mode=[2],
    ventilation_control=[0],
    verbose=False,
    confirm=True
)
output_idf = [i for i in os.listdir() if i.endswith('.idf') and i not in input_idf][0]
output_idf

Let's read it with ``load_idf()``, which reads an IDF file with eppy (with the IDD of the EnergyPlus version of the file), and let's see the subprograms that calculate its setpoints: only those of EN 16798-1.

In [ ]:
idf = accis.load_idf(output_idf)
[program.Name for program in idf.idfobjects['EnergyManagementSystem:Program'] if program.Name.startswith('SetAST_CS')]

Now, let's change the comfort standard of all its zones to ASHRAE 55 (``comfort_standard=2``), with the 80 % acceptability limits (``category=80``). The arguments that are ``None`` (by default, ``comfort_standard``, ``category``, ``comfort_mode``, ``hvac_mode`` and ``ventilation_control``) keep the values of each zone, so the comfort mode is still 3; the others take their default values, as in ``AddAccis()``.

In [ ]:
accis.modify_accis(idf, comfort_standard=2, category=80)
[program.Name for program in idf.idfobjects['EnergyManagementSystem:Program'] if program.Name.startswith('SetAST_CS')]

The model now has the EMS code of both comfort standards, and the program ``SetInputData_<zone>`` of each zone has the new arguments. ``modify_accis()`` indents the lines of the EMS programs inside the blocks ``IF ... ENDIF`` and ``WHILE ... ENDWHILE``, as ``AddAccis()`` does in the output IDFs, so that it can be seen where each block begins and ends (EnergyPlus ignores these spaces, and eppy removes them when it reads an IDF, so the programs of an IDF read with eppy are not indented until ``modify_accis()`` indents them). For instance, let's see the arguments of one of the zones, and the program ``SetComfTemp``, which calculates the comfort temperature of each comfort standard:

In [ ]:
[program for program in idf.idfobjects['EnergyManagementSystem:Program'] if program.Name.startswith('SetInputData_')][0]

In [ ]:
idf.getobject('EnergyManagementSystem:Program', 'SetComfTemp')

Finally, let's save it with the name that ``AddAccis()`` would give it, ready to be simulated:

In [ ]:
new_name = output_idf.replace('CS_INT EN16798[CA_2', 'CS_INT ASHRAE55[CA_80')
idf.saveas(new_name)
[i for i in os.listdir() if i.endswith('.idf')]

As usual, let's remove all the output idfs, so that we can re-run the notebook.

In [ ]:
for i in os.listdir():
    if i.endswith('.idf') and input_idf[0] not in i:
        os.remove(i)